###Ingestion del archivo "language.csv"

####Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
dbutils.widgets.text("p_file_date", "")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType

In [0]:
language_schema = StructType(fields =[
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

In [0]:
languages_df = spark.read \
               .option("header", True) \
               .schema(language_schema) \
                   .csv(f"abfss://bronze@moviehistory8.dfs.core.windows.net/{v_file_date}/language.csv")
               

####Paso 2 - Seleccionar solo las columnas "requeridas"

In [0]:
from pyspark.sql.functions import col

In [0]:
languages_selected_df = languages_df.select(col("languageId"), col("languageName"))

####Paso 3 - Cambiar el nombre de las columnas segun lo requerido

In [0]:
languages_renamed_df = languages_selected_df.withColumnRenamed("languageId", "language_id") \
                                            .withColumnRenamed("languageName", "language_name")

####Paso 4 - Agregar la columna "ingestion_date" y "enviroment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
languages_final_df = languages_renamed_df.withColumn("ingestion_date", current_timestamp()) \
                                         .withColumn("env", lit("Production")) \
                                         .withColumn("file_date", lit(v_file_date))

####Paso 5 - Escribir datos en el DataLake en formato "Parquet"

In [0]:
%sql
/*DROP TABLE IF EXISTS movie_silver.languages;*/

In [0]:
#languages_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistory8.dfs.core.windows.net/languages")

In [0]:
languages_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
%sql
SELECT * FROM movie_silver.languages;

language_id,language_name,ingestion_date,env,file_date
24574,English,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24575,svenska,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24576,Deutsch,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24577,unknown,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24578,Nihongo,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24579,Français,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24580,Español,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24581,al-?arabiyyah,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24582,Latin,2026-09-26T16:17:10.670014Z,Production,2024-12-16
24583,Khémôrôphéasa,2026-09-26T16:17:10.670014Z,Production,2024-12-16


In [0]:
%sql
SHOW TABLES IN movie_silver;

database,tableName,isTemporary
movie_silver,languages,false
,_sqldf,true


In [0]:
#%fs
#ls abfss://silver@moviehistory8.dfs.core.windows.net/languages

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistory8.dfs.core.windows.net/languages"))

In [0]:
display(languages_final_df)

language_id,language_name,ingestion_date,env,file_date
24574,English,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24575,svenska,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24576,Deutsch,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24577,unknown,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24578,Nihongo,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24579,Français,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24580,Español,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24581,al-?arabiyyah,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24582,Latin,2026-09-26T16:17:14.330988Z,Production,2024-12-16
24583,Khémôrôphéasa,2026-09-26T16:17:14.330988Z,Production,2024-12-16


In [0]:
dbutils.notebook.exit("Success")